# Mini-Project Temperature Lab

## Testing your specimens at different temperatures

This notebook is how you do **Part II** of your team's Shoggoth Field Journal. You take three of
your specimens and run them again and again, at different **temperature** settings, and record
what changes.

### What you'll do

1. Paste in a specimen prompt.
2. Run it at five temperatures: **0.01, 0.5, 0.9, 1.2, 1.5**.
3. Run each temperature **three times**.
4. Copy the results into your team's Journal.

That is 15 runs per specimen, and 45 runs in total. Split them across the team so everyone sees
what happens.

**NOTE:** This notebook runs the model you met as **Model A** in Worksheet 3.1, the one that was
trained to be a helpful assistant. It runs inside the notebook, so there is no account and no key
to set up.

**Your specimen may not reproduce here, and that is fine.** You found it on a much bigger AI,
like ChatGPT, Claude or Gemini. This one is far smaller, so it may answer differently. Part II is
not about making the strange behavior come back. It is about what happens to this model's
answers when you change the temperature, and that is for you to find out.

## A. Setup

Run this cell first. It takes about a minute the first time, and you only need to run it once
each time you open the notebook.

In [ ]:
#@title Run this cell first (takes about a minute)

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# The notebook asks Colab for a T4 GPU. If Colab has none to give, it still works on the CPU,
# just more slowly.
_device = "cuda" if torch.cuda.is_available() else "cpu"
if _device == "cpu":
    print("⚠️  No GPU today, so the AI will answer more slowly.")
    print("    Try Runtime > Change runtime type > T4 GPU, then run this cell again.")
    print("    (Pick T4 GPU, not TPU: this notebook cannot use a TPU.)\n")

print("Loading the AI (about 1 GB, one time only)... ", end="")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).float().to(_device)   # full precision: the model is small enough that it costs little
model.eval()
print("Done!")

# ============================================================
# FUNCTION 0: Ask the AI a question
# ============================================================
def ask_ai(prompt, temperature=1.0, max_words=50):
    """Ask the AI a question and print its answer."""
    messages = [{"role": "user", "content": prompt}]
    enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True).to(_device)
    settings = dict(max_new_tokens=max_words, pad_token_id=tokenizer.eos_token_id)
    if temperature <= 0.01:
        settings["do_sample"] = False
    else:
        settings.update(do_sample=True, temperature=temperature, top_k=0, top_p=1.0)
    with torch.no_grad():
        out = model.generate(**enc, **settings)
    answer = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print(answer.strip())

print("\n" + "=" * 50)
print("The AI is ready! Functions available:")
print("  - ask_ai(prompt, temperature=1.0)")
print("  - new_chat(temperature=1.0) / say(message) / replay_chat(temperature) / show_chat()")
print("=" * 50)

# ============================================================
# FUNCTION 4: Multi-turn chat (the AI remembers the conversation)
# ============================================================
_chat_history = []
_chat_temperature = 1.0

def _reply(history, temperature, max_words):
    """The AI's next reply to a conversation, at one temperature."""
    enc = tokenizer.apply_chat_template(history, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True).to(_device)
    settings = dict(max_new_tokens=max_words, pad_token_id=tokenizer.eos_token_id)
    if temperature <= 0.01:
        settings["do_sample"] = False
    else:
        settings.update(do_sample=True, temperature=temperature, top_k=0, top_p=1.0)
    with torch.no_grad():
        out = model.generate(**enc, **settings)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def new_chat(temperature=1.0):
    """Start a fresh conversation at one temperature. The AI forgets everything said before."""
    global _chat_history, _chat_temperature
    _chat_history = []
    _chat_temperature = temperature
    print(f"New chat started, at temperature {temperature}.")

def say(message, max_words=60):
    """Say something to the AI. It remembers everything earlier in this chat."""
    global _chat_history
    _chat_history = _chat_history + [{"role": "user", "content": message}]
    reply = _reply(_chat_history, _chat_temperature, max_words)
    _chat_history = _chat_history + [{"role": "assistant", "content": reply}]
    print("You:", message)
    print("AI:", reply)

def replay_chat(temperature, max_words=60):
    """Send the same messages again, in the same order, at a new temperature.
    The AI writes new replies, and this replay becomes the current chat."""
    global _chat_history, _chat_temperature
    messages = [turn["content"] for turn in _chat_history if turn["role"] == "user"]
    if not messages:
        print("There is no conversation to replay yet. Use new_chat() and say() first.")
        return
    _chat_history = []
    _chat_temperature = temperature
    print(f"Replaying {len(messages)} messages at temperature {temperature}.\n")
    for message in messages:
        _chat_history = _chat_history + [{"role": "user", "content": message}]
        reply = _reply(_chat_history, temperature, max_words)
        _chat_history = _chat_history + [{"role": "assistant", "content": reply}]
        print("You:", message)
        print("AI:", reply)
        print()

def show_chat():
    """Print the whole conversation so far, to copy into your Journal."""
    print("=" * 50)
    print(f"FULL CONVERSATION (temperature {_chat_temperature})")
    print("=" * 50)
    for turn in _chat_history:
        who = "You" if turn["role"] == "user" else "AI"
        print(f"\n{who}: {turn['content']}")

## B. Quick review from Worksheet 2.2

To send the AI a message, you use `ask_ai`:

In [ ]:
ask_ai("What is Quantitative Reasoning? Respond in two sentences.")

To change the temperature, you add it after the prompt:

In [ ]:
ask_ai("What is Quantitative Reasoning? Respond in two sentences.", temperature=1.2)

## C. Test your specimens

For each specimen: paste your prompt into Step 1, then run each temperature cell **three times**,
and copy the results into your Journal as you go.

### Specimen 1

**Step 1:** Paste your prompt.

In [ ]:
specimen_1_prompt = """
[PASTE YOUR SPECIMEN 1 PROMPT HERE]

Respond in 2-3 sentences.
"""

**Step 2:** Run each of these cells three times.

In [ ]:
ask_ai(specimen_1_prompt, temperature=0.01)

In [ ]:
ask_ai(specimen_1_prompt, temperature=0.5)

In [ ]:
ask_ai(specimen_1_prompt, temperature=0.9)

In [ ]:
ask_ai(specimen_1_prompt, temperature=1.2)

In [ ]:
ask_ai(specimen_1_prompt, temperature=1.5)

### Specimen 2

**Step 1:** Paste your prompt.

In [ ]:
specimen_2_prompt = """
[PASTE YOUR SPECIMEN 2 PROMPT HERE]

Respond in 2-3 sentences.
"""

**Step 2:** Run each of these cells three times.

In [ ]:
ask_ai(specimen_2_prompt, temperature=0.01)

In [ ]:
ask_ai(specimen_2_prompt, temperature=0.5)

In [ ]:
ask_ai(specimen_2_prompt, temperature=0.9)

In [ ]:
ask_ai(specimen_2_prompt, temperature=1.2)

In [ ]:
ask_ai(specimen_2_prompt, temperature=1.5)

### Specimen 3

**Step 1:** Paste your prompt.

In [ ]:
specimen_3_prompt = """
[PASTE YOUR SPECIMEN 3 PROMPT HERE]

Respond in 2-3 sentences.
"""

**Step 2:** Run each of these cells three times.

In [ ]:
ask_ai(specimen_3_prompt, temperature=0.01)

In [ ]:
ask_ai(specimen_3_prompt, temperature=0.5)

In [ ]:
ask_ai(specimen_3_prompt, temperature=0.9)

In [ ]:
ask_ai(specimen_3_prompt, temperature=1.2)

In [ ]:
ask_ai(specimen_3_prompt, temperature=1.5)

## D. Conversations, if your specimen needs one

Some specimens come from a back and forth, not from one message. Use this part only if yours does.

`new_chat()` starts a fresh conversation, and **this is where you choose its temperature**. The
whole conversation uses that one temperature. Each `say()` sends one message, and the AI remembers
everything said earlier in that chat. So **one cell is one turn** of the conversation.

In [ ]:
new_chat(temperature=1.0)
say("Hello! I'd like to play a word game with you.")

In [ ]:
say("[YOUR NEXT MESSAGE HERE]")

In [ ]:
say("[YOUR NEXT MESSAGE HERE]")

**Need more turns? Add more cells.** Hover just below the last cell and click **+ Code**, then
type another `say("...")` line in the new cell. Add as many as your conversation needs, and run
them in order, top to bottom. Running `new_chat()` again wipes the conversation and starts over.

To see the whole conversation at once, so you can copy it into your Journal:

In [ ]:
show_chat()

**Trying the same conversation at another temperature.** You do not have to type it all again.
`replay_chat()` sends your same messages, in the same order, at the temperature you give it, and
the AI writes new replies. Run it three times at each temperature, just as you did for single
prompts.

In [ ]:
replay_chat(temperature=0.01)

**NOTE:** In a replay, the AI's replies change, but your messages do not. If one of your later
messages answered something specific the AI said the first time, it may not fit the new reply.
That is worth noticing, not fixing.

**NOTE:** This small AI is not very good at long conversations, and it loses the thread quickly.
If your specimen needs many turns, record what happens here and say so in your Journal.

## Handing it in

1. Copy your results into your team's Journal, in the tables from the project brief.
2. Share this notebook as an **Editor** with everyone in your team and the instructor.
3. Put the link to this notebook in your Journal, next to your tables.

**Acknowledgments:** Notebook created for the QRAI Mini-Project, in collaboration with Claude (see [conversation](https://claude.ai/share/654b2c64-ab66-4c34-9825-29678aa5856c)).

Current version created by Ethan C. Brown in collaboration with Claude Code.